# Connect to AI models from different platforms
Same pattern for every platform:
1. **Get an API key** from the platform's website
2. **Save it in Colab Secrets** (🔑 icon on the left) with the exact name shown, and turn on *Notebook access*
3. **Run the cell**: it reads the key with `userdata.get(...)`, sends a prompt and prints the reply

| Platform | Secret name | Where to get the key | Free? |
|---|---|---|---|
| Google Gemini | `GEMINI_API_KEY` | aistudio.google.com/apikey | Yes (free tier) |
| OpenAI (GPT) | `OPENAI_API_KEY` | platform.openai.com/api-keys | Paid |
| Anthropic (Claude) | `ANTHROPIC_API_KEY` | console.anthropic.com | Paid (separate from Claude Pro) |
| Groq (Llama, Qwen…) | `GROQ_API_KEY` | console.groq.com/keys | Yes (free tier) |
| OpenRouter (300+ models, one key) | `OPENROUTER_API_KEY` | openrouter.ai/keys | Some free models |

**Tip:** model names change often. If you see a *model not found* error, run the "list models" line in that section and copy a current name.

In [ ]:
# STEP 0: Install the client libraries cleanly to avoid dependency conflicts
!pip install --force-reinstall -q -U google-genai openai anthropic

In [ ]:
# Import "userdata": Colab's tool for reading the secrets you saved under the key icon
from google.colab import userdata

# The question we will send to every AI model.
# Stored in a variable so we write it once and reuse it everywhere.
PROMPT = "Explain what a invoice is in one sentence."
print(PROMPT)

## 1. Google Gemini

In [ ]:
# Import Google's Gen AI library (the code that knows how to talk to Gemini)
from google import genai
# Import "types": ready-made settings boxes, e.g. for temperature
from google.genai import types

# Create a "client": a connection to Google, logged in with your key.
gemini = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

# Send the request to Gemini and wait for the answer
resp = gemini.models.generate_content(
    model="gemini-3.8-flash",   # Fixed: Using a valid, current Gemini model
    contents=PROMPT,
    config=types.GenerateContentConfig(temperature=0.7),
)

print(resp.text)

In [ ]:
!pip install -q --force-reinstall --no-deps openai

## 2. OpenAI (GPT)

In [ ]:
# Import OpenAI's library
from openai import OpenAI

# Create the connection, logged in with your OpenAI key from Colab Secrets
openai_client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# Fixed: Changed from non-existent gpt-5-mini to a current model
OPENAI_MODEL = "gpt-6.1-sol"

# Send a chat request
resp = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{"role": "user", "content": PROMPT}],
)

print(resp.choices[0].message.content)

## 3. Anthropic (Claude)
Note: this needs an API key from console.anthropic.com. A Claude Pro/Max chat subscription does not include API credits.

In [ ]:
# Import Anthropic's library (the code that knows how to talk to Claude)
import anthropic

# Create the connection, logged in with your Anthropic key from Colab Secrets
claude = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

# Send the message to Claude
resp = claude.messages.create(
    model="claude-sonnet-5-5",   # which Claude model to use
    max_tokens=500,              # REQUIRED for Claude: the maximum length of the answer (~1 token = 3/4 of a word)
    ##temperature=0.7,             # creativity setting, same idea as before
    messages=[{"role": "user", "content": PROMPT}],   # same message format as OpenAI
)

# Claude's answer is a list of "content blocks"; [0].text = the text of the first block
print(resp.content[0].text)

# OPTIONAL: remove the "#" below to list the Claude models available to you
# for m in claude.models.list(): print(m.id)

## 4. Groq (open-source models, very fast, free tier)
Groq speaks the **same format as OpenAI**, so you reuse the OpenAI library and only change `base_url`.

In [ ]:
# We reuse OpenAI's library, because Groq accepts the same request format
from openai import OpenAI

# Create the connection, but pointed at Groq's servers
groq = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),        # your Groq key from Colab Secrets
    base_url="https://api.groq.com/openai/v1",   # the KEY change: send requests to Groq, not OpenAI
)

# Print every model Groq currently offers, so you can copy a valid name into the next cell
for m in groq.models.list():   # loop through each model in the list
    print(m.id)                # print its name (id)

In [ ]:
# Which model to run on Groq. If you get "model not found", copy a name printed by the cell above.
GROQ_MODEL = "openai/gpt-oss-120b"

# Exactly the same call as OpenAI, because Groq is OpenAI-compatible
resp = groq.chat.completions.create(
    model=GROQ_MODEL,                                  # which model
    messages=[{"role": "user", "content": PROMPT}],    # what we ask
    temperature=0.7,                                   # how creative
)

# Pull out the answer text (same path as OpenAI)
print(resp.choices[0].message.content)

In [ ]:
from google.colab import userdata
import os

try:
    key = userdata.get('OPENROUTER_API_KEY')
    if not key:
        print("❌ 'OPENROUTER_API_KEY' is empty or not set in Colab Secrets.")
    else:
        print("✅ 'OPENROUTER_API_KEY' found in Secrets!")
        # Let's check if the key is structurally valid (typically starts with sk-or-)
        if key.startswith("sk-or-"):
            print("✅ Key format looks correct (starts with 'sk-or-').")
        else:
            print("⚠️ Warning: OpenRouter keys usually start with 'sk-or-'. Please double check your key.")
except Exception as e:
    print(f"❌ Error accessing secret: {e}")
    print("Please click the 🔑 icon in the left sidebar, add a secret named 'OPENROUTER_API_KEY', and turn on 'Notebook access'.")

## 6. One function for every platform
Many platforms accept the OpenAI format, so one helper can call them all. Add a platform by adding one line to `PLATFORMS`.

In [ ]:
from openai import OpenAI

# A "dictionary" (lookup table): platform name -> (its secret name, its server address)
# base_url = None means "use OpenAI's default address"
PLATFORMS = {
    # name:       (secret name,          base_url)
    "openai":     ("OPENAI_API_KEY",     None),
    "groq":       ("GROQ_API_KEY",       "https://api.groq.com/openai/v1"),
    #"openrouter": ("OPENROUTER_API_KEY", "https://openrouter.ai/api/v1"),
    "gemini":     ("GEMINI_API_KEY",     "https://generativelanguage.googleapis.com/v1beta/openai/"),  # Gemini's OpenAI-compatible address
}

# "def" defines a reusable function, a mini-machine you can call again and again.
# Inputs: which platform, which model, the question, and temperature (0.7 if you don't give one)
def ask(platform, model, prompt, temperature=0.7):
    # Look up this platform's secret name and address in the table above
    secret, base_url = PLATFORMS[platform]
    # Connect to that platform with its key
    client = OpenAI(api_key=userdata.get(secret), base_url=base_url)
    # Send the question
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    # Hand the answer text back to whoever called ask()
    return r.choices[0].message.content

# Try it: ask Gemini through the shared function
#print(ask("gemini", "gemini-flash-latest", PROMPT))
# Switching platform takes one word, e.g.:
print(ask("groq", "openai/gpt-oss-120b", PROMPT))

## 7. Experiment: see temperature in action
Same prompt, run 3 times at low vs high temperature. Low → nearly identical answers. High → more varied.

In [ ]:
# A creative question, where answers can vary a lot
q = "Give me a catchy name for a procurement analytics startup. Reply with the name only."

# Try two temperatures: 0.0 (safe) and 1.5 (adventurous)
for t in [0.0, 1.5]:
    print(f"\n--- temperature = {t} ---")
    # Ask the same question 3 times at this temperature
    for i in range(3):
        resp = groq.chat.completions.create(
            model="openai/gpt-oss-20b",                   # a chat model from your Groq list
            messages=[{"role": "user", "content": q}],    # the question
            temperature=t,                                # the setting we're testing
        )
        print(resp.choices[0].message.content.strip())

# Expected: at 0.0 the names barely change; at 1.5 they vary a lot

In [ ]:
# Two prompt styles: one asks for consistency, one asks for creativity
prompts = {
    "SAFE (like low temperature)":
        "Give me the most obvious, conventional name for a procurement analytics startup. Reply with the name only.",
    "CREATIVE (like high temperature)":
        "Give me a bold, unusual, surprising name for a procurement analytics startup. Avoid common words like 'procure' or 'spend'. Reply with the name only.",
}

for label, p in prompts.items():
    print(f"\n--- {label} ---")
    for i in range(3):                       # ask 3 times to see how much it varies
        resp = claude.messages.create(
            model="claude-sonnet-5-5",
            max_tokens=50,                   # a short answer is enough
            messages=[{"role": "user", "content": p}],
        )
        print(resp.content[0].text.strip())

In [ ]:
# Save the question in a variable called q
# "Reply with the name only" keeps answers short and easy to compare
q = "Give me a catchy name for a procurement analytics startup. Reply with the name only."

# Outer loop: runs twice, with t = 0.0 (safe) the first time and t = 1.5 (adventurous) the second
for t in [0.0, 1.5]:
    # Print a heading. f"..." puts the value of t into the text; \n adds a blank line before it
    print(f"\n--- temperature = {t} ---")

    # Inner loop: range(3) gives 0, 1, 2, so we ask the same question 3 times
    for i in range(3):
        # Send the question to Groq and store the full reply in "resp"
        resp = groq.chat.completions.create(
            model="openai/gpt-oss-20b",                   # which AI model answers (a chat model from your Groq list)
            messages=[{"role": "user", "content": q}],    # the conversation: one message from you ("user") with question q
            temperature=t,                                # the setting we're testing: 0.0 first, then 1.5
        )                                                 # close the request

        # Pull out the answer text and print it:
        # choices[0] = the first answer, .message.content = its text, .strip() = remove extra spaces
        print(resp.choices[0].message.content.strip())

# Total: 2 temperatures x 3 tries = 6 answers
# Expected: at 0.0 the names barely change; at 1.5 they vary a lot

In [ ]:
# A dictionary (lookup table): label -> prompt
# Two versions of the same question: one asks for an obvious name, one for a creative name
prompts = {
    # SAFE version: asks for the most obvious, conventional answer (acts like low temperature)
    "SAFE (like low temperature)":
        "Give me the most obvious, conventional name for a procurement analytics startup. Reply with the name only.",
    # CREATIVE version: asks for something bold and bans predictable words (acts like high temperature)
    "CREATIVE (like high temperature)":
        "Give me a bold, unusual, surprising name for a procurement analytics startup. Avoid common words like 'procure' or 'spend'. Reply with the name only.",
}

# Loop through the table: .items() gives each pair -> label gets the heading, p gets the prompt text
for label, p in prompts.items():
    # Print the heading for this round, with a blank line before it
    print(f"\n--- {label} ---")

    # Ask each prompt 3 times to see how much the answers vary
    for i in range(3):
        # Send the request to Claude and store the reply in "resp"
        resp = claude.messages.create(
            model="claude-sonnet-5-5",                    # which Claude model to use
            max_tokens=50,                                # required by Claude: max answer length (~35 words, keeps it short and cheap)
            messages=[{"role": "user", "content": p}],    # send the current prompt p as your message
            # no temperature here: Claude's API removed it
        )                                                 # close the request

        # Print Claude's answer
        # content[0] = the first block of the reply, .text = its text, .strip() = remove extra spaces
        print(resp.content[0].text.strip())

# Groq: same prompt, temperature changes -> different variety
# Claude: same settings, prompt wording changes -> different variety

## Common errors
| Error | Meaning | Fix |
|---|---|---|
| `SecretNotFoundError` | Secret name doesn't match, or notebook access is off | Check the spelling in 🔑 and turn on the toggle |
| `401 / invalid api key` | Wrong or expired key | Create a new key and paste it again |
| `404 / model not found` | Model name retired or misspelled | Run the list-models line and pick a current name |
| `429 / rate limit / quota` | Free-tier limit reached, or no credits | Wait a minute, or add billing |